# 🖥️ CROCO tutorial

This tutorial will show how to run a 3D simulation with output from the CROCO model. 

We start with loading the relevant modules and the data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import xroms

import parcels
import parcels.tutorial

## Create FieldSet

The dataset loaded below comes from the output of running the [CROCO Gravitational/Overflow example](https://croco-ocean.gitlabpages.inria.fr/croco_doc/model/model.test_cases.overflow.html), without any modifications to the CROCO configuration.

We use the [`xroms`](https://xroms.readthedocs.io/) package to add the vertical coordinates (depth as a function of time and space) to the dataset.

In [ ]:
ds_fields = parcels.tutorial.open_dataset("CROCOoverflow_data/data")

# Add ROMS/CROCO vertical coordinates and grid metrics (note that we need to rename Cs_rho to Cs_r for `xroms` to work)
ds_fields = ds_fields.rename({"Cs_rho": "Cs_r"})

# ds_fields, xgrid = xroms.roms_dataset(ds_fields)
# ds_fields.xroms.set_grid(xgrid)

```{note}
Note that there is currently (Nov 2026) a compatibility issue between `xroms` version 0.6.1 and `xgcm` version 0.10.1. The following code snippet provides a compatibility adapter to work around this issue. Hopefully, this will be fixed in a future release of either package.
```

In [ ]:
from unittest.mock import patch

import xgcm

# Compatibility adapter for xroms 0.6.1 with xgcm 0.10.1.
OriginalGrid = xgcm.Grid


class XromsCompatibleGrid(OriginalGrid):
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("autoparse_metadata", False)

        if "periodic" in kwargs:
            if kwargs.pop("periodic") != []:
                raise ValueError("This adapter only supports non-periodic grids")
            kwargs.setdefault("padding", "fill")

        super().__init__(*args, **kwargs)

    def diff(self, *args, **kwargs):
        if "boundary" in kwargs:
            kwargs["padding"] = kwargs.pop("boundary")
        return super().diff(*args, **kwargs)

    def interp(self, *args, **kwargs):
        if "boundary" in kwargs:
            kwargs["padding"] = kwargs.pop("boundary")
        return super().interp(*args, **kwargs)

    def derivative(self, *args, **kwargs):
        if "boundary" in kwargs:
            kwargs["padding"] = kwargs.pop("boundary")
        return super().derivative(*args, **kwargs)

In [ ]:
with patch.object(xgcm, "Grid", XromsCompatibleGrid):
    ds_fields, xgrid = xroms.roms_dataset(ds_fields)
    ds_fields.xroms.set_grid(xgrid)

Now we create a FieldSet object using the {py:func}`parcels.convert.croco_to_sgrid()` function to first create an S-Grid compliant datatset, and then use that in  {py:func}`parcels.FieldSet.from_sgrid_conventions()` to create the FieldSet.

In [ ]:
fields = {
    "U": ds_fields["u"],
    "V": ds_fields["v"],
    "W": ds_fields["w"],
    "depth": ds_fields["z_w_psi"],
    "seafloor": ds_fields["h"],
}

ds_fset = parcels.convert.croco_to_sgrid(fields=fields, coords=ds_fields)
ds_fset.time.attrs["axis"] = "T"

fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)
fieldset.add_context("hc", ds_fields.hc.item())
# Convert the FieldSet to ChunkCachedArray objects for better performance
fieldset = fieldset.to_chunk_cached_arrays()

## Create and execute ParticleSet

Now we can use this Fieldset to advect particles as we would normally do. We first define the initial positions of the particles in the domain.

```{note}
Note that the particle depths should be provided in (negative) meters, not in sigma-coordinates.
```

In [ ]:
Y, Z = np.meshgrid(
    np.arange(5e3, 60e3, 5e3),
    np.arange(-41, 0, 5),
)
Y, Z = Y.flatten(), Z.flatten()
X = 1000 * np.ones(Y.size)
T = np.timedelta64(0, "s") * np.arange(X.size)

pset_init = parcels.ParticleSet(
    fieldset=fieldset, pclass=parcels.Particle, x=X, y=Y, z=Z, t=T
)

Because some particles will be located below the seafloor, we filter them out before creating the ParticleSet.

In [ ]:
inds = np.where(fieldset.seafloor[pset_init] != 0)

pset = parcels.ParticleSet(
    fieldset=fieldset,
    pclass=parcels.Particle,
    x=X[inds],
    y=Y[inds],
    z=Z[inds],
    t=T[inds],
)

In [ ]:
def DeleteParticle(particles, fieldset):
    any_error = particles.state >= 50
    particles[any_error].state = parcels.StatusCode.Delete


outputfile = parcels.ParticleFile(
    path="croco_particles3D.parquet",
    outputdt=np.timedelta64(3, "h"),
    mode="w",
)

pset.execute(
    [parcels.kernels.AdvectionRK2_3D, DeleteParticle],
    runtime=fieldset.time_interval.right - fieldset.time_interval.left,
    dt=np.timedelta64(15, "m"),
    output_file=outputfile,
)

## Plot trajectories

Now we plot the particle trajectories below. Note that the particles roughly follow the sloping sigma-layers.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
df = parcels.read_particlefile("croco_particles3D.parquet")

for traj in df.partition_by("particle_id", maintain_order=True):
    ax.plot(traj["y"] / 1e3, traj["z"], ".-")
ax.plot(
    fieldset.depth.grid.lat[:-1, 1] / 1e3,
    fieldset.depth.data[-1, :, :, 1].T,
    "k",
    linewidth=0.5,
)
ax.set_xlabel("Y [km]")
ax.set_ylabel("Depth [m]")
ax.set_title(
    "Particles in CROCO Gravitational/Overflow velocity field using AdvectionRK2_3D"
)
plt.tight_layout()
plt.show()

## Animate trajectories

We can also make an animation, which clearly shows the development of the gravity current

In [ ]:
import matplotlib
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection

# for interactive display of animation
plt.rcParams["animation.html"] = "jshtml"

time_step = np.timedelta64(6, "h")  # time step for animation frames

# Build the time axis for the animation.
timerange = np.arange(
    np.nanmin(df["t"]),
    np.nanmax(df["t"]) + time_step,
    time_step,
)

# figure setup
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    fieldset.depth.grid.lat[:-1, 1] / 1e3,
    fieldset.depth.data[-1, :, :, 1].T,
    "k",
    linewidth=0.5,
)
ax.set_xlabel("Y [km]")
ax.set_ylabel("Depth [m]")
cmap = matplotlib.colormaps["tab20b"]

trails = LineCollection([], linewidths=0.6, alpha=0.3)
ax.add_collection(trails)

particle_ids = df["particle_id"].unique().to_list()

# Convert the trajectory data to a time-by-particle array and assign colors.
colors = np.asarray(
    [cmap(i / max(len(particle_ids) - 1, 1)) for i in range(len(particle_ids))]
)

# Create arrays to hold the x and y positions of each particle at each time step.
y = np.full((len(particle_ids), len(timerange)), np.nan)
z = np.full((len(particle_ids), len(timerange)), np.nan)

traj = df.with_columns(
    pl.col("particle_id")
    .replace(particle_ids, range(len(particle_ids)))
    .alias("p_idx"),
    pl.int_range(pl.len()).alias("row_idx"),
)

# Map each observation to its nearest animation time index.
traj = traj.with_columns(
    pl.Series("t_idx", np.searchsorted(timerange, traj["t"].to_numpy())).alias("t_idx")
)

y[traj["p_idx"].to_numpy(), traj["t_idx"].to_numpy()] = traj["y"].to_numpy() / 1e3
z[traj["p_idx"].to_numpy(), traj["t_idx"].to_numpy()] = traj["z"].to_numpy()

# Plot first timestep
scatter = ax.scatter(y[:, 0], z[:, 0], s=10, c=colors, zorder=2)

# Set initial title
t_str = timerange[0].astype("timedelta64[h]")
title = ax.set_title(f"Particles at t = {t_str}")


def animate(i):
    t_str = timerange[i].astype("timedelta64[h]")
    title.set_text(f"Particles at t = {t_str}")

    scatter.set_offsets(np.column_stack((y[:, i], z[:, i])))

    trail_length = min(10, i)  # trails have max length of 10 time steps
    start = max(0, i - trail_length)
    y_slice = y[:, start : i + 1]
    z_slice = z[:, start : i + 1]
    trails.set_segments(np.dstack((y_slice, z_slice)))
    trails.set_color(colors)


# Create animation
anim = FuncAnimation(fig, animate, frames=len(timerange), interval=100)
plt.close(fig)
anim